In [2]:
pip install nnsight

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.5/272.5 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 79.4 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.1/82.1 kB 10.4 MB/s eta 0:00:00


In [3]:
from IPython.display import clear_output
import plotly.io as pio

pio.renderers.default = 'plotly_mimetype+notebook'

In [4]:
from nnsight import LanguageModel

model = LanguageModel('allenai/OLMo-2-0425-1B')

clear_output()

In [5]:
prompt_f = 'The nurse said that'
prompt_m = 'The doctor said that'

he_token_id = model.tokenizer(' he').input_ids[0]
she_token_id = model.tokenizer(' she').input_ids[0]

with model.trace(prompt_f):
    logits_f = model.output.logits.save() # (1, num_tokens, vocab size)
probs_f = logits_f.softmax(dim=-1) # (1, num_tokens, vocab size)

with model.trace(prompt_m):
    logits_m = model.output.logits.save() # (1, num_tokens, vocab size)
probs_m = logits_m.softmax(dim=-1) # (1, num_tokens, vocab size)

clear_output()

print(f'Prompt (F): {prompt_f} \nP(he | F): {probs_f[0, -1, he_token_id]:.3f} \nP(she | F): {probs_f[0, -1, she_token_id]:.3f}')
print(f'Prompt (M): {prompt_m} \nP(he | M): {probs_f[0, -1, he_token_id]:.3f} \nP(she | M): {probs_f[0, -1, she_token_id]:.3f}')

model.safetensors.index.json:   0%|          | 0.00/14.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/179 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

Prompt (F): The nurse said that 
P(he | F): 0.057 
P(she | F): 0.118
Prompt (M): The doctor said that 
P(he | M): 0.057 
P(she | M): 0.118


In [8]:
import torch
from sklearn.decomposition import PCA
import plotly.graph_objects as go

professions_m = ["doctor", "coder", "boss", "pilot", "lawyer", "agent"]
professions_f = ["nurse", "homemaker", "secretary", "flight attendant", "paralegal", "nanny"]

prompts_m = [f'The {profession} said that' for profession in professions_m]
prompts_f = [f'The {profession} said that' for profession in professions_f]

LAYER = 5
n = len(professions_m)

with torch.no_grad():
    with model.trace(prompts_m):
        activations_m = model.model.layers[LAYER].output[:, -3, :].save() # (n, hidden dim)
    with model.trace(prompts_f):
        activations_f = model.model.layers[LAYER].output[:, -3, :].save() # (n, hidden dim)
        
pca = PCA(n_components=2)
all_activations = torch.cat([activations_m, activations_f]).cpu().float().numpy() # (2n, hidden_dim)
low_dim_activations = pca.fit_transform(all_activations) # (2n, 2)

fig = go.Figure()

fig.add_traces([
    go.Scatter(
        x=low_dim_activations[:n, 0],
        y=low_dim_activations[:n, 1],
        mode='markers',
        marker=dict(symbol='square', color='rgba(23, 94, 84, 0.4)', size=12),
        name='m',
        hovertext=professions_m,
    ),
    go.Scatter(
        x=low_dim_activations[n:, 0],
        y=low_dim_activations[n:, 1],
        mode='markers',
        marker=dict(symbol='circle', color='rgba(127, 45, 72, 0.4)', size=12),
        name='f',
        hovertext=professions_f,
    )
])

fig.update_layout(
    template='simple_white',
    width=500,
    height=400,
    xaxis_title='PCA1',
    yaxis_title='PCA2'
)

fig.show()

In [22]:
import numpy as np
from tqdm import trange
from sklearn.linear_model import LogisticRegression

def inlp(activations_m, activations_f, n_iters=10):
    results = []
    
    for i in trange(n_iters):
        X = np.concatenate([activations_m, activations_f], axis=0)
        y = np.array(['m'] * len(activations_m) + ['f'] * len(activations_f))
        
        probe = LogisticRegression(fit_intercept=False, random_state=42, max_iter=1000)
        probe.fit(X, y)
        
        probe_accuracy = probe.score(X, y)
        results.append({
            "activations_m": activations_m.copy(),
            'activations_f': activations_f.copy(),
            'iteration': i,
            'probe_accuracy': probe_accuracy,
        })
        
        if probe_accuracy < 0.51:
            print(f'Terminate at round {i} (probe accuracy = {probe_accuracy:.2f})')
            break
        
        probe_weights = probe.coef_[0]
        probe_mag = np.dot(probe_weights, probe_weights)
        
        activations_m = activations_m - np.outer(activations_m @ probe_weights, probe_weights) / probe_mag
        activations_f = activations_f - np.outer(activations_f @ probe_weights, probe_weights) / probe_mag
        
    return results

In [23]:
from plotly.subplots import make_subplots

results = inlp(activations_m.cpu().float().numpy(), activations_f.cpu().float().numpy())
n = len(professions_m)

subplot_titles = [
    f"Round {result['iteration']} | Probe Accuracy: {result['probe_accuracy']:.0%}"
    for result in results
]

fig = make_subplots(rows=1, cols=len(results), subplot_titles=subplot_titles)

for col, result in enumerate(results, start=1):
    acts_m_snap = result['activations_m']
    acts_f_snap = result['activations_f']
    
    pca = PCA(n_components=2)
    projections = pca.fit_transform(np.concatenate([acts_m_snap, acts_f_snap], axis=0))
    
    fig.add_trace(go.Scatter(
            x=projections[:n, 0],
            y=projections[:n, 1],
            mode='markers',
            marker=dict(symbol='square', color='rgba(23, 94, 84, 0.4)', size=12),
            name='m',
            showlegend=(col==1),
            hovertext=professions_m,
        ), row=1, col=col)
    
    fig.add_trace(go.Scatter(
            x=projections[n:, 0],
            y=projections[n:, 1],
            mode='markers',
            marker=dict(symbol='circle', color='rgba(127, 45, 72, 0.4)', size=12),
            name='f',
            showlegend=(col==1),
            hovertext=professions_f,
        ), row=1, col=col)

fig.update_layout(
    template='simple_white',
    width=350*len(results),
    height=400,
    title_text='Removing gender bias with INLP',
)

fig.update_xaxes(title_text='PCA 1')
fig.update_yaxes(title_text='PCA 2', col=1)

fig.show()

 20%|██        | 2/10 [00:00<00:00, 98.56it/s]

Terminate at round 2 (probe accuracy = 0.42)
